# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
# TODO
clean = df.drop_duplicates().copy()
print(df.shape)
print(clean.shape)
log('drop_duplicates',
    'Dropped duplicates',
    clean)

(315, 5)
(300, 5)
[drop_duplicates] Dropped duplicates (     order_id           item  category  qty   price
0           0    Rain Poncho  RainGear  3.0  $12.00
1           1    foam finger   Apparel  1.0     7.5
2           2  cheese burger     Merch  1.0   $7.50
3           3   Cheeseburger      Food  NaN   $7.50
4           4  cheese burger   Apparel  1.0     7.5
..        ...            ...       ...  ...     ...
295       295    rain poncho      food  3.0     6.0
296       296  cheese burger  RainGear -1.0     6.0
297       297    Foam Finger      food  1.0     7.5
298       298    Rain Poncho     Merch  1.0   $7.50
299       299  cheese burger      food  1.0  $12.00

[300 rows x 5 columns] row(s))


### TODO 2 — clean `price` -> float

In [4]:
# TODO
clean['price'] = clean['price'].astype(str).str.strip().str.replace('$', '', regex=False).astype(float)
print('price type is:',clean['price'].dtype)
log('clean_price',
    'Stripped "$" from price and converted to float',
    clean)

price type is: float64
[clean_price] Stripped "$" from price and converted to float (     order_id           item  category  qty  price
0           0    Rain Poncho  RainGear  3.0   12.0
1           1    foam finger   Apparel  1.0    7.5
2           2  cheese burger     Merch  1.0    7.5
3           3   Cheeseburger      Food  NaN    7.5
4           4  cheese burger   Apparel  1.0    7.5
..        ...            ...       ...  ...    ...
295       295    rain poncho      food  3.0    6.0
296       296  cheese burger  RainGear -1.0    6.0
297       297    Foam Finger      food  1.0    7.5
298       298    Rain Poncho     Merch  1.0    7.5
299       299  cheese burger      food  1.0   12.0

[300 rows x 5 columns] row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
# TODO
clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')

missing_qty = clean[clean['qty'].isna()].copy()
clean = clean[clean['qty'].notna()].copy()

negative_qty = clean[clean['qty']<0].copy()
clean = clean[clean['qty']>=0].copy()

clean.shape

log('drop_missing_qty',
    'Dropped rows where quantity was missing/non-numeric',
    len(missing_qty))

log('drop_negative_qty',
    'Dropped rows where quantity was negative',
    len(negative_qty))

[drop_missing_qty] Dropped rows where quantity was missing/non-numeric (12 row(s))
[drop_negative_qty] Dropped rows where quantity was negative (13 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
from re import I
# TODO: inspect the variants, build a mapping dict, apply it, log the collapse
print(df['item'].value_counts())
ITEM_MAP = {
    'cheese burger': 'Cheeseburger',
    'foam finger': 'Foam Finger',
    'rain poncho': 'Rain Poncho',
}

clean['item'] = clean['item'].replace(ITEM_MAP)

print(clean['item'].value_counts())

log('canonicalize_item',
    'Collapsed 6 spelling variants into 3 item names with ITEM_MAP',
    clean)

item
Foam Finger      61
cheese burger    55
Cheeseburger     54
Rain Poncho      52
rain poncho      49
foam finger      44
Name: count, dtype: int64
item
Foam Finger     97
Rain Poncho     91
Cheeseburger    87
Name: count, dtype: int64
[canonicalize_item] Collapsed 6 spelling variants into 3 item names with ITEM_MAP (     order_id          item  category  qty  price
0           0   Rain Poncho  RainGear  3.0   12.0
1           1   Foam Finger   Apparel  1.0    7.5
2           2  Cheeseburger     Merch  1.0    7.5
4           4  Cheeseburger   Apparel  1.0    7.5
5           5   Foam Finger      food  3.0    6.0
..        ...           ...       ...  ...    ...
294       294   Rain Poncho     Merch  2.0    6.0
295       295   Rain Poncho      food  3.0    6.0
297       297   Foam Finger      food  1.0    7.5
298       298   Rain Poncho     Merch  1.0    7.5
299       299  Cheeseburger      food  1.0   12.0

[275 rows x 5 columns] row(s))


In [7]:
print(clean['item'].value_counts())
canonical_items = {'Cheeseburger', 'Foam Finger', 'Rain Poncho'}


item
Foam Finger     97
Rain Poncho     91
Cheeseburger    87
Name: count, dtype: int64


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [8]:
# TODO
print(df['category'].value_counts())
CATEGORY_MAP = {
    'food': 'Food',
    'rain-gear': 'Rain Gear',
    'RainGear': 'Rain Gear',
}

clean['category'] = clean['category'].replace(CATEGORY_MAP)

print(clean['category'].value_counts())

log('canonicalize_category',
    'Collapsed spelling variants into 4 category names using CATEGORY_MAP',
    clean)
canonical_categories = {'Food', 'Merch', 'Apparel', 'Rain Gear'}


category
Merch        62
Food         59
food         54
rain-gear    48
Apparel      47
RainGear     45
Name: count, dtype: int64
category
Food         95
Rain Gear    86
Merch        51
Apparel      43
Name: count, dtype: int64
[canonicalize_category] Collapsed spelling variants into 4 category names using CATEGORY_MAP (     order_id          item   category  qty  price
0           0   Rain Poncho  Rain Gear  3.0   12.0
1           1   Foam Finger    Apparel  1.0    7.5
2           2  Cheeseburger      Merch  1.0    7.5
4           4  Cheeseburger    Apparel  1.0    7.5
5           5   Foam Finger       Food  3.0    6.0
..        ...           ...        ...  ...    ...
294       294   Rain Poncho      Merch  2.0    6.0
295       295   Rain Poncho       Food  3.0    6.0
297       297   Foam Finger       Food  1.0    7.5
298       298   Rain Poncho      Merch  1.0    7.5
299       299  Cheeseburger       Food  1.0   12.0

[275 rows x 5 columns] row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [9]:
assert clean.duplicated().sum() == 0
assert clean['qty'].min() >= 1
assert clean['price'].dtype == float
assert set(clean['item'].unique()) == canonical_items
assert set(clean['category'].unique()) == canonical_categories
print('clean:', clean.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [10]:
# TODO
clean['revenue'] = (clean['qty'] * clean['price']).round(2)
clean
revenue_by_cat = clean.groupby('category')['revenue'].sum().round(2).sort_values(ascending=False)
print(revenue_by_cat)
total_revenue = round(clean['revenue'].sum(), 2)
print('total revenue is:', total_revenue)

category
Food         1656.0
Rain Gear    1512.0
Merch         856.5
Apparel       715.5
Name: revenue, dtype: float64
total revenue is: 4740.0


**What I would tell the vendor:** I would tell a vendor to stock more food, as that is the highest revenue category.

### TODO 8 — read back your log

In [11]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,drop_duplicates,Dropped duplicates,order_id item category qty p...
1,clean_price,"Stripped ""$"" from price and converted to float",order_id item category qty p...
2,drop_missing_qty,Dropped rows where quantity was missing/non-nu...,12
3,drop_negative_qty,Dropped rows where quantity was negative,13
4,canonicalize_item,Collapsed 6 spelling variants into 3 item name...,order_id item category qty p...
5,canonicalize_category,Collapsed spelling variants into 4 category na...,order_id item category qty p...


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

a: Dropping the duplicates would have changed the revenue total the most. It was the first step, so negative values need to be included, the revenue changed from 4852.50 -> 4594.5 (again, final 'real' total was 4740, but dupes dropped before negative values fixed)

b: A decision I made was to keep merch and apparel as separate categories when building the category map. This is because I chose to view apparel as anything branded that could be worn (clothing), and merch as all other branded goods. Because I view these as distinct categories, it is important to understand the differences in how they contribute to revenue. If they were grouped as one category, the total would be $1572, but you would not be able to have any resolution into what was driving that revenue.